# Notebook 1 ? Join DB

Gabungkan beberapa file CSV berdasarkan kolom ID, lalu simpan hasilnya sebagai CSV. Atur nama/path file, kolom ID, dan jenis join di sel konfigurasi. Kolom dibaca sebagai teks supaya ID seperti `00123` tetap utuh.

In [ ]:
from pathlib import Path
import pandas as pd

# KONFIGURASI ? sesuaikan dengan file Anda
FILE_UTAMA = "Input/tabel_utama.csv"
ID_UTAMA = "id"
SEP_UTAMA = ","  # Gunakan ";" jika pemisah CSV adalah titik koma

FILE_JOIN = [
    {"name": "tambahan", "path": "Input/tabel_tambahan.csv", "id": "id", "how": "left", "sep": ","},
    # {"name": "referensi", "path": "Input/referensi.csv", "id": "kode", "how": "inner", "sep": ";"},
]

OUTPUT_PATH = Path("Output/hasil_join.csv")
OUTPUT_ENCODING = "utf-8-sig"
STRIP_TEXT = True
DEDUPLICATE_IDS = False  # Simpan baris pertama jika ID berulang di setiap file
def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()

TABLES = [
    {"name": "utama", "path": FILE_UTAMA, "id": ID_UTAMA, "sep": SEP_UTAMA},
    *FILE_JOIN,
]


## Baca file CSV

Periksa ringkasan jumlah baris, kolom, ID kosong, dan ID duplikat sebelum join.

In [ ]:
def load_table(config):
    path = Path(config["path"])
    if not path.is_absolute():
        path = BASE_DIR / path
    if not path.exists():
        raise FileNotFoundError(f"File '{config['name']}' tidak ditemukan: {path}")

    df = pd.read_csv(path, sep=config.get("sep", ","), dtype="string", encoding=config.get("encoding", "utf-8-sig"))
    df.columns = df.columns.astype(str).str.strip()
    id_col = config["id"]
    if id_col not in df.columns:
        raise KeyError(f"Kolom ID '{id_col}' tidak ada di {path.name}. Kolom tersedia: {df.columns.tolist()}")

    if STRIP_TEXT:
        for col in df.select_dtypes(include=["string", "object"]).columns:
            df[col] = df[col].str.strip()
    df[id_col] = df[id_col].str.strip()

    missing_ids = int(df[id_col].isna().sum() + df[id_col].eq("").sum())
    duplicate_ids = int(df.loc[df[id_col].notna() & df[id_col].ne(""), id_col].duplicated().sum())
    print(f"{config['name']}: {len(df):,} baris, {len(df.columns)} kolom, {missing_ids:,} ID kosong, {duplicate_ids:,} ID duplikat")
    if DEDUPLICATE_IDS:
        before = len(df)
        df = df.drop_duplicates(subset=[id_col], keep="first")
        print(f"  Dihapus karena ID duplikat: {before - len(df):,} baris")
    return df

if not TABLES:
    raise ValueError("Konfigurasikan minimal satu file CSV.")
tables = {config["name"]: load_table(config) for config in TABLES}
display(pd.DataFrame([{"tabel": name, "baris": len(df), "kolom": len(df.columns)} for name, df in tables.items()]))


## Join berdasarkan ID

Pilih `left`, `inner`, `right`, atau `outer` pada konfigurasi setiap file tambahan. Nama kolom ID boleh berbeda. ID kosong di file tambahan tidak dianggap cocok. Jika ada nama kolom selain ID yang sama, kolom tambahan diberi akhiran nama tabel.

In [ ]:
main_config = TABLES[0]
result = tables[main_config["name"]].copy()
main_id = main_config["id"]

for config in TABLES[1:]:
    right = tables[config["name"]].copy()
    right_id = config["id"]
    how = config.get("how", "left")
    if how not in {"left", "inner", "right", "outer"}:
        raise ValueError(f"Jenis join tidak dikenal untuk '{config['name']}': {how}")

    right = right.loc[right[right_id].notna() & right[right_id].ne("")]
    overlap = (set(result.columns) & set(right.columns)) - {main_id, right_id}
    suffixes = ("", f"_{config['name']}")
    if right_id != main_id:
        right = right.rename(columns={right_id: main_id})
    result = result.merge(right, on=main_id, how=how, suffixes=suffixes, sort=False)
    print(f"Join {config['name']} ({how}) via '{main_id}': {len(result):,} baris; kolom bertabrakan: {sorted(overlap)}")

display(result.head())
print(f"Ukuran hasil: {result.shape[0]:,} baris x {result.shape[1]} kolom")


## Simpan hasil join

In [ ]:
output_path = OUTPUT_PATH if OUTPUT_PATH.is_absolute() else BASE_DIR / OUTPUT_PATH
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path, index=False, encoding=OUTPUT_ENCODING)
print(f"Tersimpan: {output_path.resolve()}")